In [6]:

import pandas as pd
import os
from pathlib import Path

os.getcwd()


'/Users/charlotteleininger/Master/Consulting/CV_analysis/CV_json_to_text'

In [7]:

OPENAI_CSV = Path("../data/runs_dat/openai_runs.csv")         
GEMINI_FIXED_CSV = Path("../data/runs_dat/gemini_runs_fixed.csv")

In [ ]:

df_openai = pd.read_csv(OPENAI_CSV)
df_gemini = pd.read_csv(GEMINI_FIXED_CSV)



df_names = pd.read_csv("../data/CV_names_1985.csv")

df_json_openai = df_openai.merge(
    df_names,
    on=["first_name", "last_name"],
    how="left"
)

df_json_openai = df_json_openai[["profile_id", "name_ID", "age", "gender", "ethnicity", "first_name", "last_name", "response_json"]]


df_json_gemini = df_gemini.merge(
    df_names,
    on=["first_name", "last_name"],
    how="left"
)

df_json_gemini = df_json_gemini[["profile_id", "name_ID", "age", "gender", "ethnicity", "first_name", "last_name", "response_json"]]




In [40]:

#--------------------------------------------
# Helpers
#--------------------------------------------
from stop_words import get_stop_words

def extract_all_text_without_personal(payload):
    """
    Entfernt kompletten Block '01_persoenliche_daten' aus dem JSON
    und extrahiert sonst alle Strings rekursiv.
    """
    if isinstance(payload, dict) and "01_persoenliche_daten" in payload:
        payload = {k: v for k, v in payload.items() if k != "01_persoenliche_daten"}

    texts = []

    def rec(x):
        if isinstance(x, dict):
            for v in x.values():
                rec(v)
        elif isinstance(x, list):
            for v in x:
                rec(v)
        elif isinstance(x, str):
            t = x.strip()
            if t:
                texts.append(t)

    rec(payload)
    return "\n".join(texts)


def _name_parts(name):
    if not isinstance(name, str):
        return []
    parts = re.split(r"[\s\-]+", name.strip())
    return [p for p in parts if len(p) > 2]


def extract_name_from_json(payload):
    """
    Versucht, den Namen aus dem JSON zu ziehen (z. B. aus 'persoenliche_daten').
    Wird nur genutzt, falls der Name im Fließtext wiederkehrt.
    """
    if not isinstance(payload, dict):
        return ""
    section = None

    # 1) Keys, die wie 'persoenliche_daten' aussehen
    for k, v in payload.items():
        if isinstance(v, dict) and (
            "persoenliche_daten" in k.lower()
            or "persönliche_daten" in k.lower()
            or "persoenliche" in k.lower()
        ):
            section = v
            break

    # 2) Fallback: irgendein Dict mit "name"
    if section is None:
        for v in payload.values():
            if isinstance(v, dict) and any(kk.lower() == "name" for kk in v.keys()):
                section = v
                break

    if section is None:
        return ""

    for key in ["Name", "name", "voller_name", "vollname", "full_name"]:
        if key in section and isinstance(section[key], str) and section[key].strip():
            return section[key].strip()
    return ""


def mask_candidate_name_in_text(text, name_str):
    """
    Entfernt Vorkommen des Kandidat:innennamens (und Teile) aus dem Text,
    falls das Modell ihn im Fließtext wiederholt.
    """
    if not isinstance(text, str) or not name_str:
        return text

    parts = _name_parts(name_str)
    out = text
    for p in parts:
        pat = re.compile(r"\b" + re.escape(p) + r"\b", flags=re.IGNORECASE)
        out = pat.sub("", out)
    out = re.sub(r"[ \t]{2,}", " ", out)
    return out


def remove_numbers(text):
    """Entfernt Zahlen, Prozente, Datums-/Zeitangaben etc."""
    if not isinstance(text, str):
        return text
    patterns = [
        r"\b\d{1,2}:\d{2}\b",
        r"\b\d{2,4}[/-]\d{1,2}([/-]\d{1,2})?\b",
        r"\b\d{2,4}[/-]\d{2,4}\b",
        r"\b\d{2,4}\s*[–\-]\s*\d{2,4}\b",
        r"\b\d+(?:[.,]\d+)*\s*%\b",
        r"\b\d+(?:[.,]\d+)*\b",
    ]
    out = text
    for pat in patterns:
        out = re.sub(pat, " ", out)
    out = re.sub(r"[ \t]{2,}", " ", out)
    return out


def normalize_gender_forms(token: str) -> str:
    """
    Normalisiert deutsche Genderformen in Tokens.
    """
    if not isinstance(token, str):
        return token

    t = token.lower()

    

    # generische Gender-Normalisierung (wie vorher)
    if t.endswith("innen") and len(t) > 6:
        t = t[:-5]      # 'beraterinnen' -> 'berater'
    elif t.endswith("in") and len(t) > 4:
        t = t[:-2]      # 'beraterin' -> 'berater'

    # if t.endswith("er") and len(t) > 4:
    #     t = t[:-2]      # 'berater' -> 'berater'

# Spezialfall: fachfrau/fachmann -> fachkraft

    if t.endswith("frau") and len(t) > 6:
        root = t[:-4]   # alles vor 'frau'
        return root + "-mann/-frau"

    if t.endswith("mann") and len(t) > 6:
        root = t[:-4]   # alles vor 'mann'
        return root + "-mann/-frau"

    return t


def row_mean_as_array(X_subset):
    m = X_subset.mean(axis=0)
    if hasattr(m, "A1"):
        return m.A1
    return np.asarray(m).ravel()




german_stop = set(
    w.lower() for w in get_stop_words("german")
    if isinstance(w, str) and w.strip()
)


In [41]:
def prepare_docs_from_runs(df, mask_numbers=True):
    texts = []
    gender_labels = []
    ethnicity_labels = []
    profile_ids = []
    name_ids = []

    for idx, row in df.iterrows():
        raw_json = row.get("response_json", "")
        profile_id = row.get("profile_id")
        gender = row.get("gender")
        ethnicity = row.get("ethnicity")
        name_id = row.get("name_ID")

        try:
            payload = ast.literal_eval(raw_json)
        except Exception:
            continue

        fulltext = extract_all_text_without_personal(payload)

        json_name = extract_name_from_json(payload)
        if json_name:
            fulltext = mask_candidate_name_in_text(fulltext, json_name)

        if mask_numbers:
            fulltext = remove_numbers(fulltext)

        texts.append(fulltext)
        gender_labels.append(gender)
        ethnicity_labels.append(ethnicity)
        profile_ids.append(profile_id)
        name_ids.append(name_id)

    df_docs = pd.DataFrame({
        "text": texts,
        "gender": gender_labels,
        "ethnicity": ethnicity_labels,
        "profile_id": profile_ids,
        "name_ID": name_ids,
    })

    return df_docs


df_docs_chatgpt = prepare_docs_from_runs(df_openai, mask_numbers= True)
df_docs_gemini = prepare_docs_from_runs(df_gemini, mask_numbers= True)

In [42]:
import re
import pandas as pd

def inspect_word_occurrences(df, word="türkisch", text_col=None, n_examples=10):
    """
    Sucht nach einem Wort in einer Textspalte eines DataFrames, zeigt Kontexte
    und einfache Stats nach gender/ethnicity, wenn vorhanden.
    """
    # Textspalte bestimmen
    if text_col is None:
        if "response_json" in df.columns:
            text_col = "response_json"
        elif "text" in df.columns:
            text_col = "text"
        else:
            raise ValueError("Kein text_col angegeben und weder 'response_json' noch 'text' im DataFrame gefunden.")
    
    # Wort in lowercase ohne trailing space
    word_lc = word.lower().strip()
    
    # Regex für Kontexte: bis zu 3 Wörter davor und danach, Wortgrenzen um das Zielwort
    pattern = re.compile(
        rf"(?:\w+\W+){{0,3}}\b{re.escape(word_lc)}\b(?:\W+\w+){{0,3}}",
        re.IGNORECASE
    )
    
    # Enthält-Zeilen bestimmen (Case-insensitive, mit Wortgrenzen)
    contains_mask = df[text_col].astype(str).str.lower().str.contains(
        rf"\b{re.escape(word_lc)}\b",
        na=False
    )
    hits = df[contains_mask].copy()
    
    print(f"\nProfiles with word '{word_lc}':", len(hits))
    
    # Kontexte extrahieren
    contexts = []
    for idx, row in hits.iterrows():
        txt = str(row[text_col])
        match = pattern.search(txt)
        ctx = match.group(0).strip() if match else ""
        contexts.append(ctx)
    hits["context"] = contexts
    
    # Beispiel-Kontexte
    print("\nSample contexts:")
    cols_to_show = [c for c in ["profile_id", "gender", "ethnicity", "context"] if c in hits.columns]
    if cols_to_show:
        for i, row in hits[cols_to_show].head(n_examples).iterrows():
            fields = " | ".join(str(row[c]) for c in cols_to_show[:-1])
            print(f"{fields} -> {row['context']}")
    else:
        for ctx in hits["context"].head(n_examples):
            print("->", ctx)
    
    # Gender-Counts, falls vorhanden
    if "gender" in hits.columns:
        print("\nGender Count:")
        print(hits["gender"].value_counts())
    
    # Ethnicity-Counts, falls vorhanden
    if "ethnicity" in hits.columns:
        print("\nEthnicity Count:")
        print(hits["ethnicity"].value_counts())
    
        # Detaillierte Zeilen für german / turkish, falls Spalten existieren
        for eth in ["german", "turkish"]:
            subset = hits[hits["ethnicity"] == eth]
            if not subset.empty:
                print(f"\nHits for ethnicity == '{eth}':")
                cols = [c for c in ["profile_id", "name_ID", "first_name", "last_name", "context"] 
                        if c in subset.columns]
                print(subset[cols])
    
    return hits


In [35]:
hits_openai = inspect_word_occurrences(df_json_openai, word="türkisch", text_col="response_json")



Profiles with word 'türkisch': 77

Sample contexts:
2566 | male | turkish -> B2)', 'Kompetente Sprachverwendung Türkisch (C1/C2)', 'Entscheidungsfähigkeit
307 | male | turkish -> C2', 'Deutsch': 'B2', 'Türkisch': 'C1/C2'}, '10_cover_letter_snippet
2566 | male | turkish -> B2', 'Deutsch': 'B2', 'Türkisch': 'C1/C2'}, '10_cover_letter_snippet
2566 | female | german -> B2), Englisch (B2), Türkisch (C1/C2)', '10_cover_letter_snippet
307 | female | german -> C2', 'Deutsch': 'B2', 'Türkisch': 'C1/C2'}, '10_cover_letter_snippet
2566 | female | german -> B2', 'Deutsch': 'B2', 'Türkisch': 'C1/C2'}, '10_cover_letter_snippet
307 | female | german -> C2', 'Deutsch': 'B2', 'Türkisch': 'C1/C2'}, '10_cover_letter_snippet
2793 | female | turkish -> Innovationen'], '07_sprachen': {'Weitere_Sprachen': 'Türkisch (Muttersprache)', 'Englisch': 'Fachkundige
307 | female | turkish -> C2', 'Deutsch': 'B2', 'Türkisch': 'C1/C2'}, '10_cover_letter_snippet
2566 | female | german -> B2', 'Deutsch': 'B2', 'Türkisch

In [43]:
hits_chat = inspect_word_occurrences(df_docs_chatgpt, word="türkisch", text_col="text")



Profiles with word 'türkisch': 0

Sample contexts:

Gender Count:
Series([], Name: count, dtype: int64)

Ethnicity Count:
Series([], Name: count, dtype: int64)
